In [4]:
import os
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from linearmodels.panel import PanelOLS, RandomEffects
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FastICA
from sklearn.cross_decomposition import CCA
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.inspection import permutation_importance
import xgboost as xgb
from arch import arch_model
import ruptures as rpt
import networkx as nx
import hdbscan
import dowhy
from dowhy import CausalModel
import pymc as pm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

warnings.filterwarnings('ignore')
np.random.seed(42)

df = pd.read_csv('cleaned_masterdataset.csv')
df.columns = df.columns.str.strip()
df = df.rename(columns={'CAR (%)': 'CAR(%)'})

df['BANK'] = df['BANK'].astype(str).str.strip().str.upper()
df['BANK'] = df['BANK'].replace('NAN', np.nan)
df['Ownership'] = df['Ownership'].astype(str).str.strip().str.upper()

df['Year'] = df['Year'].astype(str).str.extract(r'(\d{4})')[0].astype(float)
df = df.dropna(subset=['BANK', 'Year'])
df['Year'] = df['Year'].astype(int)

num_cols = ['Gross NPA (%)', 'Net NPA (%)', 'Provision Coverage Ratio (%)', 
            'CAR(%)', 'Tier 1 Capital', 'Net Profit (cr)']

for col in num_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '', regex=True)
        df[col] = df[col].str.replace(r'^\((.*)\)$', r'-\1', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

if len(df) > 0:
    imputer = IterativeImputer(random_state=42)
    df[num_cols] = imputer.fit_transform(df[num_cols])

df = df.sort_values(by=['BANK', 'Year']).reset_index(drop=True)

df['RWA_Proxy'] = df['Tier 1 Capital'] / (df['CAR(%)'] / 100 + 1e-6)
df['NPA_CAR_Interaction'] = df['Gross NPA (%)'] * df['CAR(%)']

for col in num_cols:
    df[f'{col}_Lag1'] = df.groupby('BANK')[col].shift(1)
    df[f'{col}_RollMean3'] = df.groupby('BANK')[col].rolling(3, min_periods=1).mean().reset_index(0, drop=True)

df = df.bfill().ffill()

sys_dir = './analysis/systemic/'
os.makedirs(sys_dir, exist_ok=True)
df.to_csv(f'{sys_dir}/cleaned_data.csv', index=False)

banks = df['BANK'].unique()
for bank in banks:
    bank_dir = f'./analysis/bank/{bank}/'
    os.makedirs(bank_dir, exist_ok=True)
    df[df['BANK'] == bank].to_csv(f'{bank_dir}/cleaned_data.csv', index=False)

desc_stats = df.groupby('BANK')[num_cols].agg(['mean', 'std', 'skew', 'kurt'])
cv_func = lambda x: np.std(x, ddof=1) / np.mean(x) if np.mean(x) != 0 else np.nan
cv_stats = df.groupby('BANK')[num_cols].agg(cv_func).rename(columns=lambda x: f"{x}_CV")

own_agg = df.groupby(['Year', 'Ownership'])[num_cols].mean().unstack()
own_agg.to_csv(f'{sys_dir}/ownership_descriptive.csv')

for bank in banks:
    b_df = df[df['BANK'] == bank].copy()
    b_df[num_cols].describe().to_csv(f'./analysis/bank/{bank}/descriptive.csv')
    b_df.set_index('Year')[num_cols].pct_change().to_csv(f'./analysis/bank/{bank}/yoy_growth.csv')
    
    cagr_res = {}
    yrs = b_df['Year'].max() - b_df['Year'].min()
    if yrs > 0:
        for c in num_cols:
            start, end = b_df[c].iloc[0], b_df[c].iloc[-1]
            if start > 0 and end > 0:
                cagr_res[c] = (end/start)**(1/yrs) - 1
            else:
                cagr_res[c] = np.nan
    pd.Series(cagr_res).to_csv(f'./analysis/bank/{bank}/cagr.csv')

df['RAROC_Proxy'] = df['Net Profit (cr)'] / (df['RWA_Proxy'] + 1e-6)
sys_gnpa_mean, sys_gnpa_std = df['Gross NPA (%)'].mean(), df['Gross NPA (%)'].std()

df['Stress_Flag_2Sig'] = (df['Gross NPA (%)'] > sys_gnpa_mean + 2*sys_gnpa_std).astype(int)
df['Stress_Flag_3Sig'] = (df['Gross NPA (%)'] > sys_gnpa_mean + 3*sys_gnpa_std).astype(int)

df.groupby('BANK')[['RAROC_Proxy', 'Stress_Flag_2Sig', 'Stress_Flag_3Sig']].mean().to_csv(f'{sys_dir}/systemic_risk.csv')

for bank in banks:
    df[df['BANK'] == bank][['Year', 'RAROC_Proxy', 'Stress_Flag_2Sig', 'Stress_Flag_3Sig']].to_csv(f'./analysis/bank/{bank}/risk_metrics.csv', index=False)

try:
    panel_df = df.set_index(['BANK', 'Year'])
    exog = sm.add_constant(panel_df[['Gross NPA (%)', 'Provision Coverage Ratio (%)', 'CAR(%)']])
    endog = panel_df['Net Profit (cr)']
    
    fe_model = PanelOLS(endog, exog, entity_effects=True).fit()
    re_model = RandomEffects(endog, exog).fit()
    with open(f'{sys_dir}/panel_regression_FE.csv', 'w') as f: f.write(fe_model.summary.as_csv())
    with open(f'{sys_dir}/panel_regression_RE.csv', 'w') as f: f.write(re_model.summary.as_csv())
except Exception:
    pass

try:
    model_causal = CausalModel(
        data=df,
        treatment='Gross NPA (%)',
        outcome='Net Profit (cr)',
        common_causes=['CAR(%)', 'Tier 1 Capital', 'Provision Coverage Ratio (%)']
    )
    identified_estimand = model_causal.identify_effect()
    estimate = model_causal.estimate_effect(identified_estimand, method_name="backdoor.linear_regression")
    pd.DataFrame({'Causal_Estimate': [estimate.value]}).to_csv(f'{sys_dir}/causal_effects.csv', index=False)
except Exception:
    pass

for bank in banks:
    b_df = df[df['BANK'] == bank]
    if len(b_df) > 10:
        try:
            algo = rpt.Pelt(model="rbf").fit(b_df['Gross NPA (%)'].values)
            pd.DataFrame({'Breakpoints': algo.predict(pen=10)}).to_csv(f'./analysis/bank/{bank}/structural_breaks.csv', index=False)
        except Exception:
            pass

gnpa_pivot = df.pivot(index='Year', columns='BANK', values='Gross NPA (%)').bfill().ffill()
try:
    var_model = VAR(gnpa_pivot).fit(1)
    pd.DataFrame(var_model.forecast(gnpa_pivot.values[-1:], steps=3), columns=gnpa_pivot.columns).to_csv(f'{sys_dir}/var_forecast.csv')
except Exception:
    pass

def train_lstm(data, steps=3):
    if len(data) < 10: return np.zeros(steps)
    scaler = StandardScaler()
    scaled = scaler.fit_transform(data.reshape(-1, 1))
    X, y = [], []
    for i in range(len(scaled)-3):
        X.append(scaled[i:i+3, 0])
        y.append(scaled[i+3, 0])
    if len(X) == 0: return np.zeros(steps)
    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))
    
    model = Sequential([LSTM(10, activation='relu', input_shape=(3, 1)), Dense(1)])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=10, verbose=0)
    
    preds = []
    curr = scaled[-3:].reshape((1, 3, 1))
    for _ in range(steps):
        p = model.predict(curr, verbose=0)[0][0]
        preds.append(p)
        curr = np.append(curr[:, 1:, :], [[[p]]], axis=1)
    return scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()

for bank in banks:
    b_df = df[df['BANK'] == bank]
    try:
        lstm_f = train_lstm(b_df['Net Profit (cr)'].values)
        pd.DataFrame({'Year': [b_df['Year'].max()+i for i in range(1,4)], 'Forecast_NetProfit': lstm_f}).to_csv(f'./analysis/bank/{bank}/forecast.csv', index=False)
        
        am = arch_model(b_df['Gross NPA (%)'].pct_change().dropna() * 100, vol='Garch', p=1, q=1)
        res = am.fit(disp='off')
        pd.DataFrame(res.conditional_volatility).to_csv(f'./analysis/bank/{bank}/garch_vol.csv')
    except Exception:
        pass

try:
    pca_data = df[num_cols].dropna()
    pca = PCA(n_components=3).fit(pca_data)
    pd.DataFrame(pca.components_, columns=num_cols).to_csv(f'{sys_dir}/factors_pca.csv')
    
    ica = FastICA(n_components=3).fit(pca_data)
    pd.DataFrame(ica.components_, columns=num_cols).to_csv(f'{sys_dir}/factors_ica.csv')
    
    cca = CCA(n_components=1).fit(pca_data[['Gross NPA (%)']], pca_data[['Net Profit (cr)']])
    pd.DataFrame({'X_weights': cca.x_weights_.flatten(), 'Y_weights': cca.y_weights_.flatten()}).to_csv(f'{sys_dir}/factors_cca.csv')
    
    gmm = GaussianMixture(n_components=3).fit(pca_data)
    df['GMM_Cluster'] = gmm.predict(pca_data)
    
    hdb = hdbscan.HDBSCAN(min_cluster_size=3).fit(pca_data)
    df['HDBSCAN_Cluster'] = hdb.labels_
    df[['BANK', 'Year', 'GMM_Cluster', 'HDBSCAN_Cluster']].to_csv(f'{sys_dir}/clusters.csv', index=False)
    
    iso = IsolationForest(contamination=0.05).fit(pca_data)
    df['Anomaly_Score'] = iso.decision_function(pca_data)
    df[['BANK', 'Year', 'Anomaly_Score']].to_csv(f'{sys_dir}/anomalies.csv', index=False)
    
    X_rf = pca_data.drop(columns=['Net Profit (cr)'])
    y_rf = pca_data['Net Profit (cr)']
    rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_rf, y_rf)
    imp = permutation_importance(rf, X_rf, y_rf, n_repeats=5, random_state=42)
    
    pd.DataFrame({'Feature': X_rf.columns, 'Importance': imp.importances_mean}).to_csv(f'{sys_dir}/ml_feature_importance.csv', index=False)
    
    for bank in banks:
        b_df = df[df['BANK'] == bank]
        if len(b_df) > 0:
            preds = rf.predict(b_df[X_rf.columns])
            pd.DataFrame({'Year': b_df['Year'], 'Predicted_Profit': preds}).to_csv(f'./analysis/bank/{bank}/ml_predictions.csv', index=False)
except Exception:
    pass

try:
    corr_mat = gnpa_pivot.corr().fillna(0)
    G = nx.from_pandas_adjacency(corr_mat)
    
    cent_eigen = nx.eigenvector_centrality_numpy(G)
    cent_betw = nx.betweenness_centrality(G)
    
    pd.DataFrame({'Eigenvector': cent_eigen, 'Betweenness': cent_betw}).to_csv(f'{sys_dir}/network_centrality.csv')
    
    shock_vector = np.zeros(len(banks))
    shock_vector[0] = 1.0 
    propagation = np.dot(corr_mat.values, shock_vector)
    pd.DataFrame({'BANK': banks, 'Shock_Impact': propagation}).to_csv(f'{sys_dir}/shock_propagation.csv', index=False)
except Exception:
    pass

try:
    bank_idx = pd.factorize(df['BANK'])[0]
    coords = {"bank": df['BANK'].unique(), "obs_id": np.arange(len(df))}
    
    with pm.Model(coords=coords) as hierarchical_model:
        bank_idx_pm = pm.Data("bank_idx", bank_idx, mutable=False)
        x = pm.Data("x", df['Gross NPA (%)'].values, mutable=False)
        y = pm.Data("y", df['Net Profit (cr)'].values, mutable=False)
        
        mu_a = pm.Normal("mu_a", mu=0., sigma=100)
        sigma_a = pm.HalfNormal("sigma_a", sigma=100)
        a = pm.Normal("a", mu=mu_a, sigma=sigma_a, dims="bank")
        
        b = pm.Normal("b", mu=0., sigma=10)
        eps = pm.HalfCauchy("eps", beta=10)
        
        mu = a[bank_idx_pm] + b * x
        y_like = pm.Normal("y_like", mu=mu, sigma=eps, observed=y, dims="obs_id")
        
        trace = pm.sample(500, tune=500, cores=1, return_inferencedata=True, progressbar=False)
    
    pm.summary(trace).to_csv(f'{sys_dir}/bayesian_hierarchical.csv')
except Exception:
    pass

mc_sims = 1000
try:
    mc_results = np.zeros((mc_sims, len(banks)))
    var_res = VAR(gnpa_pivot).fit(1)
    cov_mat = var_res.sigma_u
    for i in range(mc_sims):
        shock = np.random.multivariate_normal(np.zeros(len(banks)), cov_mat)
        mc_results[i, :] = gnpa_pivot.iloc[-1].values + shock
    
    prob_collapse = (mc_results > sys_gnpa_mean + 3*sys_gnpa_std).mean(axis=0)
    pd.DataFrame({'BANK': banks, 'Prob_Collapse': prob_collapse}).to_csv(f'{sys_dir}/probabilistic_stress.csv', index=False)
except Exception:
    pass

try:
    heatmap = df.pivot_table(index='BANK', columns='Year', values='Stress_Flag_2Sig', fill_value=0)
    heatmap.to_csv(f'{sys_dir}/risk_heatmap.csv')
except Exception:
    pass

E0000 00:00:1772304974.051967   22971 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1772304974.073706   22971 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
